# Clase 195 — MLflow Tracking + Model Registry

Trackear runs (params, metrics, modelo) y promover el mejor a `Production`.

Requiere: `pip install mlflow scikit-learn`. Abrí en paralelo `mlflow ui` desde el mismo directorio.

## Setup

In [ ]:
import os, tempfile, shutil
from pathlib import Path
WORK = Path(tempfile.gettempdir()) / 'mlflow_demo'
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(); os.chdir(WORK)

import mlflow
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

mlflow.set_tracking_uri(f'file:{WORK}/mlruns')
mlflow.set_experiment('housing-demo')

X, y = fetch_california_housing(return_X_y=True, as_frame=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
print('train:', Xtr.shape, 'test:', Xte.shape)

## 1. Tracking manual — 1 run, 1 modelo

Las 3 primitivas: `log_param`, `log_metric`, `log_model`.

In [ ]:
with mlflow.start_run(run_name='lr-baseline'):
    m = LinearRegression().fit(Xtr, ytr)
    pred = m.predict(Xte)
    mlflow.log_param('model_type', 'LinearRegression')
    mlflow.log_metric('rmse_test', np.sqrt(mean_squared_error(yte, pred)))
    mlflow.log_metric('r2_test', r2_score(yte, pred))
    mlflow.sklearn.log_model(m, name='model')
    print('run_id:', mlflow.active_run().info.run_id)

## 2. Autolog — sin boilerplate

`autolog` engancha el `.fit()` y registra params + metrics + modelo automáticamente.

In [ ]:
mlflow.sklearn.autolog(log_models=True, log_datasets=False)
with mlflow.start_run(run_name='rf-autolog'):
    RandomForestRegressor(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42).fit(Xtr, ytr)
    # rmse_test no se loguea solo; lo agregamos a mano si lo querés en el dashboard
mlflow.sklearn.autolog(disable=True)

## 3. Sweep — 10 runs variando hiperparámetros

In [ ]:
from itertools import product
for depth, n_est in product([3, 5, 10, 15, 20], [50, 200]):
    with mlflow.start_run(run_name=f'rf_d{depth}_n{n_est}'):
        m = RandomForestRegressor(n_estimators=n_est, max_depth=depth, n_jobs=-1, random_state=42).fit(Xtr, ytr)
        rmse = np.sqrt(mean_squared_error(yte, m.predict(Xte)))
        mlflow.log_params({'model_type': 'RandomForest', 'max_depth': depth, 'n_estimators': n_est})
        mlflow.log_metric('rmse_test', rmse)
        mlflow.log_metric('r2_test', r2_score(yte, m.predict(Xte)))
        mlflow.sklearn.log_model(m, name='model')
print('10 runs registrados.')

## 4. Búsqueda + Registry

Encontramos el mejor por `rmse_test` y lo registramos como `housing-best`.

In [ ]:
runs = mlflow.search_runs(order_by=['metrics.rmse_test ASC'], filter_string="params.model_type = 'RandomForest'")
best = runs.iloc[0]
print('mejor run:', best['run_id'], '| rmse:', round(best['metrics.rmse_test'], 4))

result = mlflow.register_model(f"runs:/{best['run_id']}/model", 'housing-best')
print('version registrada:', result.version)

In [ ]:
# Transicionar a Production (API legacy — sigue funcionando)
from mlflow.tracking import MlflowClient
client = MlflowClient()
client.transition_model_version_stage(name='housing-best', version=result.version, stage='Production', archive_existing_versions=True)

# API moderna (MLflow 2.5+): alias
client.set_registered_model_alias('housing-best', 'champion', result.version)
print('promoted to Production + alias @champion')

## 5. Carga "en producción"

Un servicio cualquiera carga el modelo por URI lógica — no necesita saber el `run_id`.

In [ ]:
loaded = mlflow.pyfunc.load_model('models:/housing-best@champion')
preds = loaded.predict(Xte.head())
print('predicciones:', preds)
print('reales:      ', yte.head().values)

## Ejercicio guiado

1. Agregá un run con `XGBRegressor` (`pip install xgboost`) y compará vs el mejor RF.
2. Usá `mlflow.search_runs` con filtro `"metrics.r2_test > 0.7"` y ordená por `rmse_test`.
3. Promové un challenger con alias `@challenger` y escribí una celda que cargue ambos (`@champion`, `@challenger`) y compare predicciones sobre las primeras 100 filas de test.

## Conclusiones

- `log_param`/`log_metric`/`log_model` son la API mínima viable; **autolog** quita boilerplate cuando el framework lo soporta.
- El **Model Registry** desacopla `entrenar` de `servir`: la API en producción referencia `models:/housing-best@champion`, no un `run_id`.
- Promover de `champion` a otro modelo es **una llamada API**, no un redeploy del servidor.

## ✅ Soluciones de los ejercicios

Soluciones de los 5 ejercicios del README. MLflow no está instalado en este entorno, así que **cada solución muestra la llamada real de MLflow** (`mlflow.start_run`, `log_param`, `log_metric`, `register_model`, `transition_model_version_stage`, `pyfunc.load_model`) y, para que la celda **corra igual**, ejecuta el *concepto* con un tracker/registry mínimo en memoria que imita la semántica de MLflow. Así ves qué guarda MLflow por debajo: params + metrics por run, y un registry con stages.

In [ ]:
# --- tracker mínimo que imita la semántica de MLflow (para que corra sin mlflow) ---
class MiniRun:
    def __init__(self, store, name):
        self.store, self.name = store, name
        self.data = {'name': name, 'params': {}, 'metrics': {}}
    def __enter__(self):
        return self
    def __exit__(self, *a):
        self.store.append(self.data)
    def log_param(self, k, v): self.data['params'][k] = v
    def log_metric(self, k, v): self.data['metrics'][k] = v

class MiniMlflow:
    def __init__(self): self.runs = []
    def start_run(self, run_name=''): return MiniRun(self.runs, run_name)
    def search_runs(self):
        import pandas as pd
        rows = [{'name': r['name'], **{f'params.{k}': v for k, v in r['params'].items()},
                 **{f'metrics.{k}': v for k, v in r['metrics'].items()}} for r in self.runs]
        return pd.DataFrame(rows)

mlf = MiniMlflow()
print('tracker listo (imita mlflow.*).')

### Ejercicio 1 — Tracking manual de dos modelos

API real de MLflow (en comentario). Abrimos un run por modelo y logueamos hiperparámetros + RMSE train/test. En MLflow los verías lado a lado en `mlflow ui`.

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

X, y = fetch_california_housing(return_X_y=True, as_frame=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
rmse = lambda yt, yp: mean_squared_error(yt, yp) ** 0.5

# API REAL de mlflow:
#   import mlflow, mlflow.sklearn
#   with mlflow.start_run(run_name="linreg"):
#       mlflow.log_param("model", "LinearRegression")
#       mlflow.log_metric("rmse_test", rmse_test)
#       mlflow.sklearn.log_model(model, "model")
for name, model in [('linreg', LinearRegression()),
                    ('rf', RandomForestRegressor(n_estimators=100, max_depth=10, random_state=0))]:
    model.fit(Xtr, ytr)
    with mlf.start_run(run_name=name) as run:
        if hasattr(model, 'n_estimators'):
            run.log_param('n_estimators', model.n_estimators)
        run.log_param('max_depth', getattr(model, 'max_depth', None))
        run.log_metric('rmse_train', round(rmse(ytr, model.predict(Xtr)), 4))
        run.log_metric('rmse_test', round(rmse(yte, model.predict(Xte)), 4))

print(mlf.search_runs()[['name', 'metrics.rmse_train', 'metrics.rmse_test']].to_string(index=False))
assert len(mlf.runs) == 2
print('\nOK — 2 runs trackeados (en MLflow: mlflow ui --port 5000 para compararlos).')

### Ejercicio 2 — Autolog

`mlflow.sklearn.autolog()` intercepta `.fit()` y loguea params + metrics + el modelo **sin código extra**. Lo simulamos capturando `model.get_params()` automáticamente.

In [ ]:
# API REAL:
#   mlflow.sklearn.autolog()   # a partir de aquí, cada .fit() se loguea solo
#   with mlflow.start_run():
#       RandomForestRegressor(max_depth=8).fit(Xtr, ytr)   # params+metrics+model auto

def autolog_fit(model, name):
    model.fit(Xtr, ytr)
    with mlf.start_run(run_name=name) as run:
        for k, v in model.get_params().items():      # <- lo que hace autolog: TODOS los params
            run.log_param(k, v)
        run.log_metric('rmse_test', round(rmse(yte, model.predict(Xte)), 4))
    return model

autolog_fit(RandomForestRegressor(max_depth=8, n_estimators=50, random_state=0), 'rf_autolog')
last = mlf.runs[-1]
assert 'max_depth' in last['params'] and 'rmse_test' in last['metrics']
print('autolog capturó', len(last['params']), 'params y', len(last['metrics']), 'metric sin código extra.')

### Ejercicio 3 — Sweep de hiperparámetros + `search_runs`

10 combinaciones de `max_depth × n_estimators`. `mlflow.search_runs()` devuelve un DataFrame; filtramos por `rmse_test` mínimo.

In [ ]:
sweep = MiniMlflow()
for md in [3, 5, 10, 15, 20]:
    for ne in [50, 200]:
        m = RandomForestRegressor(max_depth=md, n_estimators=ne, random_state=0).fit(Xtr, ytr)
        with sweep.start_run(run_name=f'md{md}_ne{ne}') as run:
            run.log_param('max_depth', md); run.log_param('n_estimators', ne)
            run.log_metric('rmse_test', round(rmse(yte, m.predict(Xte)), 4))

df = sweep.search_runs()
# equivalente a: mlflow.search_runs(order_by=["metrics.rmse_test ASC"]).iloc[0]
best = df.loc[df['metrics.rmse_test'].idxmin()]
print(df.sort_values('metrics.rmse_test').to_string(index=False))
assert len(df) == 10
print(f'\nmejor: {best["name"]}  rmse_test={best["metrics.rmse_test"]}')

### Ejercicio 4 — Model Registry: register + transition de stages

Registramos el mejor modelo y lo movemos `None → Staging → Production` con la API del `MlflowClient`. Simulamos el registry (un dict versión→stage).

In [ ]:
# API REAL:
#   result = mlflow.register_model(model_uri, "housing-rf")
#   client = mlflow.tracking.MlflowClient()
#   client.transition_model_version_stage("housing-rf", result.version, stage="Staging")
#   client.transition_model_version_stage("housing-rf", result.version, stage="Production")

class MiniRegistry:
    def __init__(self): self.models = {}
    def register(self, name, model):
        v = len(self.models.get(name, [])) + 1
        self.models.setdefault(name, []).append({'version': v, 'stage': 'None', 'model': model})
        return v
    def transition(self, name, version, stage):
        for mv in self.models[name]:
            if mv['version'] == version: mv['stage'] = stage
    def get_by_stage(self, name, stage):
        return next(mv['model'] for mv in self.models[name] if mv['stage'] == stage)

best_model = RandomForestRegressor(max_depth=15, n_estimators=200, random_state=0).fit(Xtr, ytr)
registry = MiniRegistry()
v = registry.register('housing-rf', best_model)
registry.transition('housing-rf', v, 'Staging')
registry.transition('housing-rf', v, 'Production')
print('housing-rf v%d -> stage:' % v, registry.models['housing-rf'][0]['stage'])
assert registry.models['housing-rf'][0]['stage'] == 'Production'
print('OK — modelo promovido a Production en el registry.')

### Ejercicio 5 — Carga en "producción" y predicción

Un servicio carga el modelo `Production` por su URI y predice una fila. En MLflow: `mlflow.pyfunc.load_model("models:/housing-rf/Production")`.

In [ ]:
# API REAL:
#   model = mlflow.pyfunc.load_model("models:/housing-rf/Production")
#   pred = model.predict(X_one)
prod_model = registry.get_by_stage('housing-rf', 'Production')   # <- resuelve el URI al modelo Production
X_one = Xte.iloc[[0]]
pred = prod_model.predict(X_one)
print('predicción (median house value, $100k):', round(float(pred[0]), 3))
print('valor real:', round(float(yte.iloc[0]), 3))
assert pred.shape == (1,)
print('OK — el servicio sirve SIEMPRE el modelo marcado Production (desacoplado del training).')